# CNN Practical Assignment
## Oxford-IIIT Pet Dataset

**Student:** Alankar Sunil Uniyal  
**Course:** Deep Learning  
**Submission:** Final Report

This notebook presents the final report for the multi-task CNN assignment on the Oxford-IIIT Pet Dataset. The model implementation, training pipeline, evaluation utilities, checkpoints, and generated outputs are maintained in the accompanying Git repository.

The notebook is report-oriented. It documents the experimental design, architecture, preprocessing, losses, training configuration, quantitative results, qualitative results, and the interpretation of the three required augmentation experiments.

**Repository URL:** `https://github.com/auniyal26/dl_assignment2`

---

# Question 1 — Multi-Task CNN for Classification and Semantic Segmentation

**Task.** Develop a single convolutional neural network that simultaneously:

1. predicts one of the **37 pet breeds**, and  
2. predicts a **binary pet/background segmentation mask**.

The model must use a shared CNN encoder followed by separate classification and segmentation branches, must be trained jointly, and must not use a pretrained network or an existing segmentation architecture directly.

## 1. Dataset, Split, and Preprocessing

The Oxford-IIIT Pet dataset provides RGB images, 37 breed labels, and pixel-level trimaps. The official `trainval` split contains **3,680** samples and the official `test` split contains **3,669** samples.

A fixed random seed of **42** was used to split the official `trainval` set into:

| Split | Number of Samples | Source |
|---|---:|---|
| Training | 2,944 | 80% of official `trainval` |
| Validation | 736 | 20% of official `trainval` |
| Test | 3,669 | official `test` split |

The exact same train/validation/test indices were reused for all three augmentation experiments so that augmentation strategy remained the controlled experimental factor.

### Image Preprocessing

All images were resized to **128 × 128** pixels using bilinear interpolation. Each image was then converted to a tensor and normalized channel-wise using

$$
\mu = (0.5, 0.5, 0.5), \qquad
\sigma = (0.5, 0.5, 0.5)
$$

which maps image tensors approximately from $[0,1]$ to $[-1,1]$.

### Segmentation-Mask Preprocessing

The Oxford-IIIT Pet trimap uses the values:

- `1`: pet
- `2`: background
- `3`: boundary

The assignment requires binary semantic segmentation. Therefore the trimap was converted using

$$
M_{\text{binary}}
=
\mathbb{1}
\left[
M_{\text{trimap}} \neq 2
\right]
$$

so values `1` and `3` are treated as foreground and value `2` as background.

Masks were resized using **nearest-neighbour interpolation** rather than bilinear interpolation. This preserves discrete label values and avoids introducing invalid intermediate mask values.

### Annotation Anomaly Check

Seven samples in the official `trainval` split and seven samples in the official test split contain all-background trimaps despite visibly containing pets. These masks were confirmed to be present in the raw dataset rather than being introduced by preprocessing.

The anomalous test-mask indices are:

`[1093, 1690, 1858, 2292, 2837, 2856, 3473]`

These samples were retained in the **official full-test metrics** so that the test set was not selectively modified. They were excluded only when deliberately selecting qualitative success/failure examples, because an invalid ground-truth mask should not be presented as a model failure. A valid-mask-only sensitivity analysis is reported later.

### Answer

The final split is **2,944 training / 736 validation / 3,669 test images**. Images are resized to $128 \times 128$, converted to tensors, and normalized to approximately $[-1,1]$. Trimaps are converted to binary masks with values `1` and `3` as foreground and value `2` as background, using nearest-neighbour interpolation for masks.

## 2. Architecture

A custom multi-task encoder-decoder network was implemented from scratch. The model contains four shared convolutional encoder blocks, a classification branch, and a four-stage segmentation decoder. Skip connections pass encoder features to the corresponding decoder stages.

Each `ConvBlock` contains two $3 \times 3$ convolutions, each followed by a ReLU activation. All convolutions use padding 1, preserving spatial resolution within a block. A $2 \times 2$ max-pooling operation reduces spatial resolution between encoder blocks.

### Complete Architecture

| Component | Input | Operation | Output | Activation |
|---|---|---|---|---|
| Encoder Block 1 | $3 \times 128 \times 128$ | 2 × Conv $3 \times 3$, 32 channels | $32 \times 128 \times 128$ | ReLU |
| MaxPool | $32 \times 128 \times 128$ | $2 \times 2$ | $32 \times 64 \times 64$ | — |
| Encoder Block 2 | $32 \times 64 \times 64$ | 2 × Conv $3 \times 3$, 64 channels | $64 \times 64 \times 64$ | ReLU |
| MaxPool | $64 \times 64 \times 64$ | $2 \times 2$ | $64 \times 32 \times 32$ | — |
| Encoder Block 3 | $64 \times 32 \times 32$ | 2 × Conv $3 \times 3$, 128 channels | $128 \times 32 \times 32$ | ReLU |
| MaxPool | $128 \times 32 \times 32$ | $2 \times 2$ | $128 \times 16 \times 16$ | — |
| Encoder Block 4 | $128 \times 16 \times 16$ | 2 × Conv $3 \times 3$, 256 channels | $256 \times 16 \times 16$ | ReLU |
| Bottleneck Pool | $256 \times 16 \times 16$ | $2 \times 2$ | $256 \times 8 \times 8$ | — |
| Classification GAP | $256 \times 8 \times 8$ | Adaptive global average pooling | $256 \times 1 \times 1$ | — |
| Classification FC | 256 | Linear 256 → 128 | 128 | ReLU + Dropout(0.30) |
| Classification Output | 128 | Linear 128 → 37 | 37 logits | — |
| Decoder Up 4 | $256 \times 8 \times 8$ | Transposed Conv $2 \times 2$, 128 | $128 \times 16 \times 16$ | — |
| Skip + Decoder Block 4 | $(128 + 256)$ channels | 2 × Conv $3 \times 3$, 128 | $128 \times 16 \times 16$ | ReLU |
| Decoder Up 3 | $128 \times 16 \times 16$ | Transposed Conv $2 \times 2$, 64 | $64 \times 32 \times 32$ | — |
| Skip + Decoder Block 3 | $(64 + 128)$ channels | 2 × Conv $3 \times 3$, 64 | $64 \times 32 \times 32$ | ReLU |
| Decoder Up 2 | $64 \times 32 \times 32$ | Transposed Conv $2 \times 2$, 32 | $32 \times 64 \times 64$ | — |
| Skip + Decoder Block 2 | $(32 + 64)$ channels | 2 × Conv $3 \times 3$, 32 | $32 \times 64 \times 64$ | ReLU |
| Decoder Up 1 | $32 \times 64 \times 64$ | Transposed Conv $2 \times 2$, 32 | $32 \times 128 \times 128$ | — |
| Skip + Decoder Block 1 | $(32 + 32)$ channels | 2 × Conv $3 \times 3$, 32 | $32 \times 128 \times 128$ | ReLU |
| Segmentation Output | $32 \times 128 \times 128$ | Conv $1 \times 1$, 1 channel | $1 \times 128 \times 128$ logits | — |

The classification branch uses global average pooling to convert the shared encoder representation into a compact feature vector without requiring a very large fully connected layer.

The segmentation branch uses transposed convolutions for learned upsampling and skip connections to restore higher-resolution spatial information lost during encoder pooling.

In [ ]:
# Architecture verification only; this cell performs no training.
try:
    from model import MultiTaskPetCNN

    model = MultiTaskPetCNN()
    trainable_parameters = sum(
        p.numel() for p in model.parameters() if p.requires_grad
    )
    print(f"Trainable parameters: {trainable_parameters:,}")
except Exception as exc:
    print("Could not import the repository model:", exc)
    print("Expected trainable parameters: 2,188,646")

### Trainable Parameters

The implemented model contains

$$
\boxed{2,188,646 \text{ trainable parameters}}
$$

### Answer

The final model uses four encoder blocks and four decoder blocks, satisfying the required minimum of three convolutional encoder blocks and three decoder blocks. It is trained from scratch, uses a shared encoder, produces 37 classification logits, and reconstructs a full-resolution binary segmentation mask. No pretrained model or existing segmentation architecture is used directly.

## 3. Loss Functions

### Classification Loss

Breed classification uses multi-class cross-entropy:

$$
L_{\text{cls}}
=
-\frac{1}{N}
\sum_{i=1}^{N}
\log p_{i,y_i}
$$

This is appropriate because each image belongs to exactly one of 37 mutually exclusive breed classes.

### Segmentation Loss

The segmentation objective combines binary cross-entropy with Dice loss:

$$
L_{\text{seg}}
=
0.5L_{\text{BCE}}
+
0.5L_{\text{Dice}}
$$

`BCEWithLogitsLoss` is applied directly to the raw segmentation logits for numerical stability.

For soft probabilities $p_i = \sigma(z_i)$, the Dice coefficient is

$$
D
=
\frac{
2\sum_i p_i y_i + \epsilon
}{
\sum_i p_i + \sum_i y_i + \epsilon
}
$$

and the corresponding Dice loss is

$$
L_{\text{Dice}} = 1 - D
$$

BCE provides pixel-wise supervision, while Dice directly encourages overlap between the predicted foreground region and the target foreground region.

### Joint Multi-Task Loss

The classification and segmentation losses are combined using equal task-level weights:

$$
L_{\text{total}}
=
1.0L_{\text{cls}}
+
1.0L_{\text{seg}}
$$

Within the segmentation term, BCE and Dice each receive weight 0.5.

### Answer

The classification task uses cross-entropy loss. The segmentation task uses an equal combination of BCE-with-logits and Dice loss. The two task losses are then added with equal task-level weights of 1.0.

## 4. Training Configuration

The final controlled experiment used the following configuration for **all three augmentation conditions**:

| Setting | Value |
|---|---|
| Optimizer | Adam |
| Initial learning rate | $1 \times 10^{-3}$ |
| Batch size | 32 |
| Maximum epochs | 30 |
| Random seed | 42 |
| Image resolution | $128 \times 128$ |
| Classification-loss weight | 1.0 |
| Segmentation-loss weight | 1.0 |
| LR scheduler | `ReduceLROnPlateau` |
| Scheduler monitor | joint validation loss |
| LR reduction factor | 0.5 |
| Scheduler patience | 2 epochs |
| Relative threshold | $1 \times 10^{-3}$ |
| Minimum learning rate | $1 \times 10^{-5}$ |
| Checkpoint selection | lowest joint validation loss |

The scheduler configuration was identical across all three experiments. Because `ReduceLROnPlateau` is adaptive, the realized learning-rate trajectory can differ between experiments even though the training policy is unchanged.

### Training-Protocol Refinement

A preliminary 20-epoch diagnostic run showed that the augmented models were still improving near the end of training, while the non-augmented model was beginning to show a larger train-validation separation. The final experiment therefore used a common maximum of 30 epochs together with an adaptive learning-rate schedule.

The final test evaluation uses the checkpoint with the **lowest joint validation loss**, rather than automatically using the final epoch. This avoids selecting a checkpoint using test-set information.

### Training and Validation Loss

![Training and validation total loss](outputs/report_30epoch/loss_curves_30epoch.png)

### Learning-Rate Schedule

![Learning-rate schedule](outputs/report_30epoch/learning_rate_schedule_30epoch.png)

### Answer

All three experiments use the same architecture, split, optimizer, initial learning rate, batch size, maximum epoch count, loss formulation, scheduler configuration, and checkpoint-selection rule. This preserves augmentation strategy as the principal controlled experimental factor.

## 5. Baseline Classification Performance

For Question 1, the **No Augmentation** model is treated as the baseline multi-task CNN.

The baseline test classification accuracy is

$$
\boxed{0.140093 \approx 14.01\%}
$$

The classification task contains 37 fine-grained pet breeds and the network is trained entirely from scratch, without pretrained visual features.

### Baseline Confusion Matrix

![Baseline confusion matrix](outputs/report_30epoch/no_augmentation_confusion_matrix_30epoch.png)

### Answer

The baseline model achieves **14.01% test classification accuracy**. The confusion matrix above shows the class-level distribution of correct and incorrect predictions.

## 6. Baseline Segmentation Performance

Binary predictions are obtained by applying a sigmoid to the segmentation logits and thresholding at 0.5.

For a predicted binary mask $P$ and a ground-truth mask $G$, Intersection over Union is

$$
\text{IoU}
=
\frac{|P \cap G|}{|P \cup G|}
$$

and Dice score is

$$
\text{Dice}
=
\frac{2|P \cap G|}{|P| + |G|}
$$

The baseline test results are:

| Metric | Full Official Test Set | Valid-Mask-Only Sensitivity Check |
|---|---:|---:|
| Mean IoU | **0.760719** | 0.762173 |
| Mean Dice | **0.853994** | 0.855626 |

The valid-mask-only values differ from the full-test values by only about 0.0015–0.0016, indicating that the seven anomalous masks have negligible influence on the aggregate conclusion.

### Answer

The baseline segmentation branch achieves **mean IoU = 0.7607** and **mean Dice = 0.8540** on the full official test split.

## 7. Baseline Qualitative Segmentation Analysis

The following figure contains three strong and two weak segmentation examples selected only from test samples with valid foreground annotations.

![Baseline segmentation examples](outputs/report_30epoch/no_augmentation_segmentation_examples_30epoch.png)

Successful cases generally contain a clearly visible pet occupying a substantial portion of the frame and having a relatively distinct object boundary.

Difficult cases tend to involve one or more of the following:

- a small or distant pet,
- unusual framing or partial visibility,
- cluttered background structure,
- low contrast between the animal and its surroundings,
- fine boundary details lost during encoder downsampling.

### Answer

The network is strongest when the pet is large, clearly visible, and spatially separated from the background. Failures are concentrated in images with small foreground regions, clutter, unusual framing, or ambiguous boundaries.

# Question 2 — Effect of Data Augmentation

The same model architecture, dataset split, optimizer, scheduler configuration, batch size, maximum epoch count, loss formulation, and checkpoint-selection rule are used for all three experiments.

The only systematic experimental difference is the training-time augmentation strategy.

## 8. Augmentation Strategies

### Experiment A — No Augmentation

Training images receive only deterministic resizing, tensor conversion, and normalization.

![No-augmentation examples](outputs/report_30epoch/baseline_augmentation_examples.png)

### Experiment B — Geometric Augmentation

The following random transformations are applied during training:

- random resized crop with scale range 0.80–1.00,
- crop aspect-ratio range 0.90–1.10,
- random horizontal flip with probability 0.5,
- random rotation between $-15^\circ$ and $+15^\circ$.

Every sampled geometric transform is applied **consistently to both the image and the segmentation mask**. Images use bilinear interpolation and masks use nearest-neighbour interpolation.

![Geometric augmentation examples](outputs/report_30epoch/geometric_augmentation_examples.png)

### Experiment C — Geometric + Appearance Augmentation

Experiment C uses all geometric transformations from Experiment B and additionally applies image-only color jitter:

- brightness = 0.25,
- contrast = 0.25,
- saturation = 0.25.

Appearance transformations are applied **only to the image**, never to the segmentation mask.

![Geometric and appearance augmentation examples](outputs/report_30epoch/geometric_plus_appearance_augmentation_examples.png)

Validation and test images use deterministic baseline preprocessing for all three experiments.

## 9. Final Quantitative Comparison

The final test-set comparison is:

| Experiment | Classification Accuracy | Mean IoU | Mean Dice |
|---|---:|---:|---:|
| No Augmentation | 0.140093 | 0.760719 | 0.853994 |
| Geometric Augmentation | 0.245026 | **0.762329** | **0.854688** |
| Geometric + Appearance | **0.274462** | 0.751952 | 0.847445 |

In percentage form, classification accuracy changes from **14.01%** without augmentation to **24.50%** with geometric augmentation and **27.45%** with geometric plus appearance augmentation.

Relative to the baseline:

- geometric augmentation improves classification by approximately **10.49 percentage points** while maintaining segmentation quality,
- adding appearance augmentation improves classification by a further **2.94 percentage points** over geometric augmentation,
- appearance augmentation slightly reduces segmentation IoU and Dice.

### Valid-Mask Sensitivity Check

| Experiment | Valid-Mask Mean IoU | Valid-Mask Mean Dice |
|---|---:|---:|
| No Augmentation | 0.762173 | 0.855626 |
| Geometric Augmentation | **0.763786** | **0.856321** |
| Geometric + Appearance | 0.753390 | 0.849065 |

The experiment ranking is unchanged, confirming that the seven anomalous test masks do not materially affect the result.

In [ ]:
# Display the final comparison table directly from the saved experiment output.
from pathlib import Path
import pandas as pd

comparison_path = Path("outputs") / "comparison.csv"

if comparison_path.exists():
    comparison = pd.read_csv(comparison_path)
else:
    comparison = pd.DataFrame({
        "Experiment": [
            "No Augmentation",
            "Geometric Augmentation",
            "Geometric + Appearance",
        ],
        "Classification Accuracy": [0.140093, 0.245026, 0.274462],
        "Mean IoU": [0.760719, 0.762329, 0.751952],
        "Mean Dice": [0.853994, 0.854688, 0.847445],
        "Valid-mask Mean IoU": [0.762173, 0.763786, 0.753390],
        "Valid-mask Mean Dice": [0.855626, 0.856321, 0.849065],
    })

display(comparison)

## 10. Training and Validation Performance

### Classification Accuracy

![Training and validation classification accuracy](outputs/report_30epoch/accuracy_curves_30epoch.png)

### Mean IoU

![Training and validation IoU](outputs/report_30epoch/iou_curves_30epoch.png)

### Mean Dice

![Training and validation Dice](outputs/report_30epoch/dice_curves_30epoch.png)

The late-training train-validation gaps provide additional evidence about the regularizing effect of augmentation.

At epoch 30:

| Experiment | Train Accuracy | Validation Accuracy | Accuracy Gap | Train IoU | Validation IoU | IoU Gap |
|---|---:|---:|---:|---:|---:|---:|
| No Augmentation | 0.5662 | 0.2215 | **0.3448** | 0.8560 | 0.7464 | **0.1096** |
| Geometric Augmentation | 0.3387 | 0.2283 | 0.1104 | 0.7796 | 0.7552 | 0.0245 |
| Geometric + Appearance | 0.3427 | 0.2582 | **0.0846** | 0.7713 | 0.7502 | **0.0212** |

The final-epoch validation-minus-training total-loss gaps are approximately:

- **2.471** for no augmentation,
- **0.569** for geometric augmentation,
- **0.538** for geometric + appearance augmentation.

The non-augmented model therefore exhibits substantially stronger late-training overfitting.

The scheduler behavior is also informative. The baseline model reaches its lowest validation loss around epoch 17, after which its learning rate is repeatedly reduced. By contrast, both augmented models reach their best validation loss around epoch 27 and only trigger their first learning-rate reduction after epoch 30. This indicates that augmentation allows the models to continue learning useful generalizable features for longer.

In [ ]:
# Recompute the final-epoch train/validation gaps from saved histories.
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("outputs")

history_files = {
    "No Augmentation": OUTPUT_DIR / "no_augmentation_history.csv",
    "Geometric Augmentation": OUTPUT_DIR / "geometric_augmentation_history.csv",
    "Geometric + Appearance": OUTPUT_DIR / "geometric_plus_appearance_history.csv",
}

rows = []

for name, path in history_files.items():
    if not path.exists():
        continue

    h = pd.read_csv(path)
    r = h.iloc[-1]

    rows.append({
        "Experiment": name,
        "Train Accuracy": r["train_accuracy"],
        "Validation Accuracy": r["val_accuracy"],
        "Accuracy Gap": r["train_accuracy"] - r["val_accuracy"],
        "Train IoU": r["train_iou"],
        "Validation IoU": r["val_iou"],
        "IoU Gap": r["train_iou"] - r["val_iou"],
        "Validation - Training Loss": r["val_loss"] - r["train_loss"],
    })

if rows:
    display(pd.DataFrame(rows))
else:
    print("History CSV files were not found in ./outputs.")

## 11. Confusion Matrices

### No Augmentation

![No augmentation confusion matrix](outputs/report_30epoch/no_augmentation_confusion_matrix_30epoch.png)

### Geometric Augmentation

![Geometric augmentation confusion matrix](outputs/report_30epoch/geometric_augmentation_confusion_matrix_30epoch.png)

### Geometric + Appearance Augmentation

![Geometric plus appearance confusion matrix](outputs/report_30epoch/geometric_plus_appearance_confusion_matrix_30epoch.png)

The aggregate accuracies show that both augmentation strategies substantially improve breed classification relative to the baseline. The geometric-plus-appearance experiment achieves the highest overall classification accuracy.

## 12. Segmentation Examples for Augmented Models

### Geometric Augmentation

![Geometric augmentation segmentation examples](outputs/report_30epoch/geometric_augmentation_segmentation_examples_30epoch.png)

### Geometric + Appearance Augmentation

![Geometric plus appearance segmentation examples](outputs/report_30epoch/geometric_plus_appearance_segmentation_examples_30epoch.png)

The geometric-only model produces the strongest aggregate segmentation result. The appearance-augmented model still segments pets effectively overall, but its lower IoU and Dice indicate a small degradation in spatial mask quality.

# 13. Required Discussion

## 13.1 Which augmentation strategy gives the best classification performance?

**Geometric + appearance augmentation gives the best classification performance.**

Its test accuracy is

$$
\boxed{27.45\%}
$$

compared with 24.50% for geometric augmentation and 14.01% for no augmentation.

The result indicates that introducing variation in both geometry and appearance improves the model's ability to generalize breed-discriminative features to unseen test images.

### Answer

**Geometric + appearance augmentation is best for classification, with 27.45% test accuracy.**

---

## 13.2 Which augmentation strategy gives the best segmentation performance?

**Geometric augmentation gives the best segmentation performance.**

It achieves

$$
\boxed{\text{Mean IoU} = 0.762329}
$$

and

$$
\boxed{\text{Mean Dice} = 0.854688}
$$

The improvement over the baseline is small but consistent in both IoU and Dice.

### Answer

**Geometric augmentation is best for segmentation, with mean IoU 0.7623 and mean Dice 0.8547.**

---

## 13.3 Does augmentation affect classification and segmentation in the same way?

No. The effect is task-dependent.

For classification:

$$
14.01\%
\rightarrow
24.50\%
\rightarrow
27.45\%
$$

For segmentation:

$$
\text{IoU: }
0.7607
\rightarrow
0.7623
\rightarrow
0.7520
$$

Thus, the augmentation strategy that is best for classification is not the one that is best for segmentation.

A plausible explanation is that classification benefits from learning invariance to color, brightness, contrast, orientation, and framing. Segmentation depends more directly on spatially precise object/background cues. Additional appearance perturbations may therefore improve recognition while slightly weakening cues useful for localization.

### Answer

**No.** Augmentation has a substantially larger positive effect on classification. Geometric augmentation preserves segmentation quality, whereas additional appearance augmentation improves classification further but slightly reduces segmentation performance.

---

## 13.4 Does augmentation reduce the difference between training and validation performance?

Yes.

At epoch 30, the classification train-validation accuracy gap is approximately:

- no augmentation: **34.48 percentage points**,
- geometric augmentation: **11.04 percentage points**,
- geometric + appearance augmentation: **8.46 percentage points**.

The IoU train-validation gap shows the same pattern:

- no augmentation: **0.1096**,
- geometric augmentation: **0.0245**,
- geometric + appearance augmentation: **0.0212**.

The total-loss separation is also substantially larger for the baseline.

### Answer

**Yes.** Both augmentation strategies substantially reduce the late-training train-validation gap and therefore act as effective regularizers.

---

## 13.5 Are there augmentations that appear to hurt performance?

Yes, but the effect is task-specific.

Adding brightness, contrast, and saturation augmentation improves classification from 24.50% to 27.45%. However, relative to geometric augmentation, it lowers segmentation performance:

$$
\text{IoU: }
0.7623
\rightarrow
0.7520
$$

and

$$
\text{Dice: }
0.8547
\rightarrow
0.8474
$$

Therefore, appearance augmentation appears mildly harmful to segmentation under the current multi-task architecture, even though it is beneficial for breed classification.

### Answer

**Appearance augmentation mildly hurts segmentation while improving classification.** Geometric augmentation provides the best overall balance between the two tasks.

# 14. Overall Conclusion

A custom multi-task CNN was successfully trained from scratch to perform simultaneous 37-class breed classification and binary pet segmentation on the Oxford-IIIT Pet Dataset.

The main experimental findings are:

1. The shared encoder-decoder architecture is effective for segmentation, reaching approximately **0.76 mean IoU** and **0.85 mean Dice**.
2. Classification is substantially more sensitive to augmentation than segmentation.
3. Geometric augmentation increases test classification accuracy from **14.01% to 24.50%** while slightly improving segmentation metrics.
4. Adding appearance augmentation increases classification accuracy further to **27.45%**, but reduces segmentation performance to **0.7520 IoU / 0.8474 Dice**.
5. Both augmentation strategies substantially reduce the late-training train-validation gap, demonstrating a clear regularization effect.
6. The strategy that performs best for classification is not the strategy that performs best for segmentation.
7. The seven anomalous all-background test masks have negligible quantitative influence, as confirmed by the valid-mask sensitivity analysis.

The final result therefore demonstrates a genuine multi-task trade-off. Transformations that encourage classification invariance do not necessarily improve pixel-level localization.

Under the current architecture and training protocol, **geometric augmentation provides the best overall balance**, while **geometric + appearance augmentation is preferable when classification accuracy is the priority**.

# Appendix A — Implementation Reference

The complete implementation is maintained in the accompanying Git repository rather than duplicated inside this report notebook.

| File | Purpose |
|---|---|
| `config.py` | experiment constants, device configuration, and optimizer/scheduler settings |
| `data.py` | dataset loading, fixed split, preprocessing, and augmentation |
| `model.py` | multi-task CNN architecture |
| `losses_metrics.py` | classification loss, segmentation loss, IoU, and Dice |
| `train.py` | training loop, validation, adaptive learning-rate scheduling, and checkpoint selection |
| `evaluate.py` | predictions, confusion matrices, and segmentation evaluation |
| `run_all.py` | runs the three controlled experiments |
| `make_report_outputs.py` | regenerates report figures from saved histories and checkpoints |
| `outputs/*.csv` | final experiment histories and comparison table |
| `outputs/*_best.pt` | best-validation-loss checkpoints |

**Repository URL:** `ADD_GITHUB_REPOSITORY_URL_HERE`

The repository is the implementation reference for this report. This notebook itself is not used to train the final models.

---

# Appendix B — Final Result Summary

| Experiment | Classification Accuracy | Mean IoU | Mean Dice |
|---|---:|---:|---:|
| No Augmentation | 14.01% | 0.7607 | 0.8540 |
| Geometric Augmentation | 24.50% | **0.7623** | **0.8547** |
| Geometric + Appearance | **27.45%** | 0.7520 | 0.8474 |

**Best classification:** Geometric + Appearance  
**Best segmentation:** Geometric Augmentation  
**Best overall multi-task balance:** Geometric Augmentation